This notebook builds the training data: for each collection, it saves a table listing every usable field with its crop type, its train/validation/test split (taken from FTW's chips), and its AEF and OlmoEarth embeddings.

In [1]:
import json
import time
import fsspec
import shapely
import numpy as np
import pandas as pd
import pyarrow as pa
import geopandas as gpd
from pathlib import Path
import pyarrow.compute as pc
import pyarrow.parquet as pq
from urllib.parse import urljoin
from IPython.display import display
pd.set_option('display.max_rows',20)

In [2]:
CATALOGURL  = 'https://data.source.coop/ftw/benchmark-data/catalog.json'
RAWDIR      = Path('../data/raw')
INTERIMDIR  = Path('../data/interim')
PROCDIR     = Path('../data/processed')
SETTINGS    = json.loads((INTERIMDIR/'settings.json').read_text())
COLLECTIONS = pd.read_csv(INTERIMDIR/'collections.csv').collection.tolist()
MINFIELDS   = SETTINGS['minfields']
FILECOLUMNS = SETTINGS['filecolumns']
MAXFIELDS   = 200000
MINPIXELS   = 5
REMOTE      = fsspec.filesystem('https')

In [3]:
def build_file_url(collection,filetype):
    return urljoin(CATALOGURL,f"{collection}/{collection.split('/')[1]}_{filetype}.parquet")

def build_raw_path(collection,filetype):
    return RAWDIR/f"{collection.split('/')[1]}_{filetype}.parquet"

def save_raw_copy(collection,filetype):
    rawpath = build_raw_path(collection,filetype)
    if not rawpath.exists():
        partpath = rawpath.with_suffix('.part')
        with REMOTE.open(build_file_url(collection,filetype)) as file:
            parquet = pq.ParquetFile(file)
            schema  = pa.schema([parquet.schema_arrow.field(col) for col in FILECOLUMNS[filetype]],metadata=parquet.schema_arrow.metadata)
            with pq.ParquetWriter(partpath,schema,compression='zstd') as writer:
                for batch in parquet.iter_batches(batch_size=100000,columns=FILECOLUMNS[filetype]):
                    writer.write_batch(batch)
        partpath.rename(rawpath)

def scan_rows(rawpath,cols,ids):
    keep = pa.array(ids)
    for batch in pq.ParquetFile(rawpath).iter_batches(batch_size=50000,columns=cols):
        yield batch.filter(pc.is_in(batch['id'],value_set=keep))

def drop_fields(fields,keep,reason,log):
    log[reason] = int((~keep).sum())
    return fields[keep]

In [4]:
def build_table(collection):
    for filetype in FILECOLUMNS:
        save_raw_copy(collection,filetype)
    log = {'collection':collection}
    labels = pd.read_parquet(build_raw_path(collection,'fields'),columns=['id','hcat:name'],dtype_backend='pyarrow')
    pixels = pd.read_parquet(build_raw_path(collection,'field_embeddings'),columns=['id','pixels','pixel_rule'],dtype_backend='pyarrow')
    fields = labels.merge(pixels.drop_duplicates('id'),on='id',how='left').rename(columns={'hcat:name':'crop'})
    log['start'] = len(fields)
    fields = drop_fields(fields,fields.crop.notna(),'nolabel',log)
    fields = drop_fields(fields,~fields.id.duplicated(keep=False),'duplicateid',log)
    fields = drop_fields(fields,(fields.pixels>=MINPIXELS)&(fields.pixel_rule!='touched'),'small',log)
    counts = fields.crop.value_counts()
    fields = drop_fields(fields,fields.crop.map(counts)>=MINFIELDS,'rarecrop',log)
    quota  = (fields.crop.value_counts()*min(1,MAXFIELDS/len(fields))).round().clip(lower=MINFIELDS)
    fields = fields.sample(frac=1,random_state=100)
    fields = drop_fields(fields,fields.groupby('crop').cumcount()<fields.crop.map(quota),'capped',log)[['id','crop']]
    ids    = fields.id.to_numpy()
    stats      = ['aef_mean','aef_std','olmoearth_mean','olmoearth_std']
    batches    = [(batch['id'].to_numpy(zero_copy_only=False),np.hstack([batch[stat].flatten().to_numpy().reshape(batch.num_rows,-1) for stat in stats]))
                  for batch in scan_rows(build_raw_path(collection,'field_embeddings'),['id']+stats,ids)]
    featureids = np.concatenate([batchids for batchids,_ in batches])
    features   = np.vstack([values for _,values in batches])
    del batches
    chips     = gpd.read_parquet(build_raw_path(collection,'chips'))
    chipgeoms = np.asarray(chips.geometry)
    locations = []
    for batch in scan_rows(build_raw_path(collection,'fields'),['id','geometry'],ids):
        geoms     = shapely.make_valid(shapely.from_wkb(batch['geometry'].to_numpy(zero_copy_only=False)))
        fieldidx,chipidx = chips.sindex.query(geoms,predicate='intersects')
        overlap   = shapely.area(shapely.intersection(geoms[fieldidx],chipgeoms[chipidx]))
        best      = pd.Series(chipidx,index=fieldidx).iloc[np.argsort(overlap)].groupby(level=0).last()
        centroids = shapely.centroid(geoms)
        locations.append(pd.DataFrame({'id':batch['id'].to_numpy(zero_copy_only=False),'chipidx':best.reindex(range(len(geoms))).to_numpy(),
                                       'lon':shapely.get_x(centroids),'lat':shapely.get_y(centroids)}))
    location = pd.concat(locations)
    info   = pd.DataFrame({'id':featureids}).merge(fields.astype(object),on='id',how='left').merge(location,on='id',how='left')
    inchip = info.chipidx.notna().to_numpy()
    log['outsidechips'] = int((~inchip).sum())
    info   = info[inchip].reset_index(drop=True)
    chipidx = info.pop('chipidx').astype(int).to_numpy()
    info.insert(2,'chip',chips.id.to_numpy()[chipidx])
    info.insert(3,'split',chips.split.to_numpy()[chipidx])
    names  = [f'{model}{i:03d}' for model in ('aef','olmoearth') for i in range(128)]
    table  = pd.concat([info,pd.DataFrame(features[inchip],columns=names)],axis=1)
    table.to_parquet(PROCDIR/f"fields_{collection.split('/')[1]}.parquet",index=False)
    log.update(kept=len(table),crops=table.crop.nunique(),**table.split.value_counts().reindex(['train','val','test']).to_dict())
    return log

In [ ]:
todo = [collection for collection in COLLECTIONS if not (PROCDIR/f"fields_{collection.split('/')[1]}.parquet").exists()]
print(f'Building {len(todo)} of {len(COLLECTIONS)} collections...')

logs = []
for collection in todo:
    start = time.perf_counter()
    logs.append(build_table(collection))
    minutes,seconds = divmod(round(time.perf_counter()-start),60)
    print(f'- Saved {PROCDIR}/fields_{collection.split("/")[1]}.parquet ({minutes}m {seconds}s)')

logpath = INTERIMDIR/'build_log.csv'
log     = pd.concat([pd.read_csv(logpath) if logpath.exists() else None,pd.DataFrame(logs)]).drop_duplicates('collection',keep='last')
log.set_index('collection').reindex(COLLECTIONS).dropna(how='all').reset_index().to_csv(logpath,index=False)

Building 18 of 18 collections...
- Saved ../data/processed/fields_ee_2024.parquet (0m 20s)
- Saved ../data/processed/fields_sk_2024.parquet (0m 35s)
- Saved ../data/processed/fields_lv_2024.parquet (0m 29s)
- Saved ../data/processed/fields_cz_2024.parquet (0m 25s)
- Saved ../data/processed/fields_be_vlg_2024.parquet (0m 18s)
- Saved ../data/processed/fields_dk_2024.parquet (0m 23s)
- Saved ../data/processed/fields_es_cat_2024.parquet (0m 23s)
- Saved ../data/processed/fields_si_2024.parquet (0m 23s)
- Saved ../data/processed/fields_fi_2024.parquet (0m 26s)
- Saved ../data/processed/fields_ie_2024.parquet (0m 34s)
- Saved ../data/processed/fields_se_2024.parquet (0m 27s)
- Saved ../data/processed/fields_lt_2024.parquet (0m 31s)
- Saved ../data/processed/fields_hr_2024.parquet (0m 24s)
- Saved ../data/processed/fields_nl_2024.parquet (0m 39s)
- Saved ../data/processed/fields_es_vc_2024.parquet (0m 41s)
- Saved ../data/processed/fields_at_2024.parquet (0m 52s)
- Saved ../data/processed/fi

In [ ]:
log     = pd.read_csv(INTERIMDIR/'build_log.csv')
headers = {'collection':'Collection',
           'start':'Fields in Source File',
           'nolabel':'Dropped: No Crop Label',
           'duplicateid':'Dropped: Duplicate Field ID',
           'small':f'Dropped: Smaller than {MINPIXELS} Pixels',
           'rarecrop':f'Dropped: Crop with < {MINFIELDS} Fields',
           'capped':f'Dropped: Over the {MAXFIELDS:,}-Field Cap',
           'outsidechips':'Dropped: Outside Every Chip',
           'kept':'Fields Kept',
           'crops':'Crops Kept',
           'train':'Training Fields',
           'val':'Validation Fields',
           'test':'Test Fields'}

print(f'{log.kept.sum():,.0f} fields kept across {len(log)} collections')
display(log[list(headers)].rename(columns=headers).style.format('{:,.0f}',subset=list(headers.values())[1:]).hide(axis='index'))